# Training

QLoRA fine-tuning of Mistral 7B on the data from `02_data_pipeline.ipynb`. The default batch size
was chosen for a GPU with around 100 GB of memory. On an A100 40 GB add `--medium-gpu`, on an L4
add `--small-gpu`, and on a T4 add `--small-gpu --fp16`.

Add `HF_TOKEN` and `WANDB_API_KEY` to the Colab secrets (the key icon on the left) so the adapter
can be pushed to the Hugging Face Hub and the run logged to Weights & Biases.

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
if not os.path.isdir("/content/chess-llm-thesis"):
    !git clone -q https://github.com/Lefterisstam/chess-llm-thesis.git /content/chess-llm-thesis
%cd /content/chess-llm-thesis
!pip install -q -r requirements.txt
!pip install -q flash-attn --no-build-isolation

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

## Smoke test

100 steps on a small subset, to check that everything runs before the long training.

In [ ]:
!python model/training/train.py --smoke

## Stage 1

8,000 steps with a new LoRA adapter. A checkpoint is saved to Drive every 500 steps; if the session
stops, running the cell again continues from the last checkpoint.

In [ ]:
!python model/training/train.py

## Stage 2

Continues the stage 1 adapter for one full pass over the training data with half the learning rate.
This produced the final model.

In [ ]:
!python model/training/train.py --continued --packing --per-device-train-batch-size 16

## Try the new adapter

In [ ]:
ADAPTER = "/content/drive/MyDrive/chess_llm_checkpoints/mistral7b-qlora-chess-v2/final_adapter"
FEN = "r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3"
HISTORY = "e2e4 e7e5 g1f3 b8c6 f1b5"

!python model/inference/play.py --adapter {ADAPTER} --mode exact --fen "{FEN}" --history "{HISTORY}" --show-candidates